Pixeltable can send tabular data directly to the [Rerun](https://rerun.io/) viewer, providing a way to inspect dataframes alongside Rerun's powerful multimodal visualizations. In this tutorial, we'll learn how to:

- Export data from a Pixeltable table to Rerun as an Arrow RecordBatch
- Select specific columns for export
- Send data to a notebook-embedded Rerun viewer
- Send data to a remote Rerun Viewer

We begin by installing the necessary libraries.

In [1]:
%pip install -e /path/to/pixeltable rerun-sdk

ERROR: /path/to/pixeltable is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Dev-only: force reload of local pixeltable source tree
import importlib
import pixeltable.io
import pixeltable.io.globals

importlib.reload(pixeltable.io.globals)
importlib.reload(pixeltable.io)

<module 'pixeltable.io' from '/Users/pierre/pixeltable/pixeltable/io/__init__.py'>

## Example 1: Sending a Table to Rerun

Let's start by creating a simple Pixeltable table with mixed data types and sending it to Rerun.

In [3]:
import pixeltable as pxt

pxt.drop_dir('rerun_demo', force=True)
pxt.create_dir('rerun_demo')

Connected to Pixeltable database at: postgresql+psycopg://postgres:@/pixeltable?host=/Users/pjlb/.pixeltable/pgdata
Found an existing Pixeltable dashboard at: http://localhost:22089
Created directory 'rerun_demo'.


In [4]:
t = pxt.create_table(
    'rerun_demo.sensor_data',
    {
        'timestamp': pxt.Timestamp,
        'sensor_id': pxt.String,
        'temperature': pxt.Float,
        'humidity': pxt.Float,
        'status': pxt.String,
        'metadata': pxt.Json,
    },
)

Created table 'sensor_data'.


In [5]:
from datetime import datetime, timezone

t.insert(
    [
        {
            'timestamp': datetime(
                2025, 1, 15, 10, 0, 0, tzinfo=timezone.utc
            ),
            'sensor_id': 'sensor_A',
            'temperature': 22.5,
            'humidity': 45.2,
            'status': 'normal',
            'metadata': {'location': 'room_1', 'firmware': 'v2.1'},
        },
        {
            'timestamp': datetime(
                2025, 1, 15, 10, 5, 0, tzinfo=timezone.utc
            ),
            'sensor_id': 'sensor_B',
            'temperature': 25.1,
            'humidity': 52.8,
            'status': 'warning',
            'metadata': {'location': 'room_2', 'firmware': 'v2.0'},
        },
        {
            'timestamp': datetime(
                2025, 1, 15, 10, 10, 0, tzinfo=timezone.utc
            ),
            'sensor_id': 'sensor_A',
            'temperature': 23.0,
            'humidity': 44.9,
            'status': 'normal',
            'metadata': {'location': 'room_1', 'firmware': 'v2.1'},
        },
        {
            'timestamp': datetime(
                2025, 1, 15, 10, 15, 0, tzinfo=timezone.utc
            ),
            'sensor_id': 'sensor_C',
            'temperature': 19.8,
            'humidity': 60.1,
            'status': 'normal',
            'metadata': {'location': 'room_3', 'firmware': 'v2.2'},
        },
        {
            'timestamp': datetime(
                2025, 1, 15, 10, 20, 0, tzinfo=timezone.utc
            ),
            'sensor_id': 'sensor_B',
            'temperature': 26.3,
            'humidity': 55.4,
            'status': 'alert',
            'metadata': {'location': 'room_2', 'firmware': 'v2.0'},
        },
    ]
)

Inserted 5 rows with 0 errors in 0.02 s (253.77 rows/s)


5 rows inserted.

In [6]:
t.head()

timestamp,sensor_id,temperature,humidity,status,metadata
2025-01-15 02:00:00-08:00,sensor_A,22.5,45.2,normal,"{""firmware"": ""v2.1"", ""location"": ""room_1""}"
2025-01-15 02:05:00-08:00,sensor_B,25.1,52.8,warning,"{""firmware"": ""v2.0"", ""location"": ""room_2""}"
2025-01-15 02:10:00-08:00,sensor_A,23.,44.9,normal,"{""firmware"": ""v2.1"", ""location"": ""room_1""}"
2025-01-15 02:15:00-08:00,sensor_C,19.8,60.1,normal,"{""firmware"": ""v2.2"", ""location"": ""room_3""}"
2025-01-15 02:20:00-08:00,sensor_B,26.3,55.4,alert,"{""firmware"": ""v2.0"", ""location"": ""room_2""}"


### Dry Run: Inspect the Arrow RecordBatch

When called without a `viewer` or `addr`, `send_to_rerun` converts the table data to a PyArrow
RecordBatch and returns it without sending. This is useful for inspecting what will be sent.

In [7]:
batch = pxt.io.send_to_rerun(t)
print(f'Rows: {batch.num_rows}')
print(f'Schema: {batch.schema}')
batch.to_pandas()

Rows: 5
Schema: timestamp: timestamp[us, tz=America/Los_Angeles]
sensor_id: string
temperature: double
humidity: double
status: string
metadata: string


,timestamp,sensor_id,temperature,humidity,status,metadata
0,2025-01-15 02:00:00-08:00,sensor_A,22.5,45.2,normal,"{""firmware"": ""v2.1"", ""location"": ""room_1""}"
1,2025-01-15 02:05:00-08:00,sensor_B,25.1,52.8,warning,"{""firmware"": ""v2.0"", ""location"": ""room_2""}"
2,2025-01-15 02:10:00-08:00,sensor_A,23.0,44.9,normal,"{""firmware"": ""v2.1"", ""location"": ""room_1""}"
3,2025-01-15 02:15:00-08:00,sensor_C,19.8,60.1,normal,"{""firmware"": ""v2.2"", ""location"": ""room_3""}"
4,2025-01-15 02:20:00-08:00,sensor_B,26.3,55.4,alert,"{""firmware"": ""v2.0"", ""location"": ""room_2""}"


### Selecting Specific Columns

You can choose which columns to include in the export using the `columns` parameter.

In [8]:
batch = pxt.io.send_to_rerun(
    t,
    columns=[t.sensor_id, t.temperature, t.humidity],
    table_name='Sensor Readings',
)
batch.to_pandas()

,sensor_id,temperature,humidity
0,sensor_A,22.5,45.2
1,sensor_B,25.1,52.8
2,sensor_A,23.0,44.9
3,sensor_C,19.8,60.1
4,sensor_B,26.3,55.4


## Example 2: Sending to a Notebook Viewer

Rerun provides an inline notebook viewer via the `rerun_notebook` package. Pass a `Viewer` instance
to `send_to_rerun` to display the table directly in the notebook.

> **Note:** The inline viewer requires `pip install "rerun-sdk[notebook]"` and a Jupyter-compatible environment.

In [ ]:
import os
from rerun_notebook import Viewer

os.environ['RERUN_NOTEBOOK_ASSET'] = 'inline'

viewer = Viewer(width=800, height=400)
viewer.block_until_ready(timeout=10.0)
pxt.io.send_to_rerun(t, table_name='Sensor Data', viewer=viewer)
viewer

## Example 3: Sending to a Remote Rerun Viewer

If you have a Rerun Viewer running externally (launched via `rerun` in your terminal),
you can send data to it by passing the `addr` parameter.

```bash
# In a separate terminal, start the Rerun Viewer:
rerun
```

Then send data from Pixeltable:

In [ ]:
# Uncomment to send to a running Rerun Viewer:
# pxt.io.send_to_rerun(
#     t,
#     table_name='Sensor Data',
#     addr='rerun+http://0.0.0.0:9876/proxy',
# )

## Example 4: Working with Image Tables

Pixeltable tables with image columns are also supported. Image values are represented as
descriptive strings (e.g., `Image(640x480, RGB)`) in the Rerun table view.

In [ ]:
img_table = pxt.create_table(
    'rerun_demo.images', {'image': pxt.Image, 'caption': pxt.String}
)

url_prefix = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images'

img_table.insert(
    [
        {
            'image': f'{url_prefix}/000000000025.jpg',
            'caption': 'A giraffe standing in a field',
        },
        {
            'image': f'{url_prefix}/000000000030.jpg',
            'caption': 'A bowl of food on a table',
        },
        {
            'image': f'{url_prefix}/000000000034.jpg',
            'caption': 'A baseball player at bat',
        },
    ]
)

In [ ]:
batch = pxt.io.send_to_rerun(img_table, table_name='Image Catalog')
batch.to_pandas()

## Example 5: Computed Columns

Computed columns work seamlessly with the Rerun export. Add a computed column to your table
and it will be included automatically.

In [ ]:
@pxt.udf
def classify_temp(temperature: float) -> str:
    if temperature is None:
        return 'unknown'
    if temperature < 20:
        return 'cold'
    if temperature < 25:
        return 'comfortable'
    return 'hot'


t.add_computed_column(temp_class=classify_temp(t.temperature))

In [ ]:
batch = pxt.io.send_to_rerun(
    t,
    columns=[t.sensor_id, t.temperature, t.temp_class],
    table_name='Temperature Classification',
)
batch.to_pandas()

## API Summary

`pxt.io.send_to_rerun()` supports three usage modes:

| Mode | Parameter | Description |
|------|-----------|-------------|
| Notebook | `viewer=rerun_notebook.Viewer(...)` | Sends to an inline Rerun viewer |
| Remote | `addr='rerun+http://...'` | Connects to a running Rerun Viewer |
| Dry run | _(neither)_ | Returns the RecordBatch without sending |

Additional parameters:

- **`columns`**: List of column expressions to include (default: all columns)
- **`table_name`**: Name for the table entry in the Rerun viewer (default: Pixeltable table name)

For the full API reference, see [`pxt.io.send_to_rerun`](https://docs.pixeltable.com/sdk/latest/io#send_to_rerun).

## Type Handling Reference

Pixeltable column types are converted to Arrow-compatible values as follows:

| Pixeltable Type | Arrow Representation |
|-----------------|---------------------|
| `Int`, `Float`, `Bool`, `String` | Direct Arrow equivalents |
| `Timestamp`, `Date` | Arrow temporal types |
| `Json` | Serialized JSON string |
| `Image` | Description string, e.g. `Image(640x480, RGB)` |
| `Video`, `Audio`, `Document` | File path or URL string |
| `Array` | Stringified list |
| `UUID` | String representation |